# 06 · Reporting — Stage 7

Assembles `reports/final_report.html`: SCR storyline → Governing Thought → three MECE Key Lines →
recommendation, then one section per Key Line with an exhibit and a "So What", then the appendix.

**Every number in the report is read from `reports/_key_figures.json` and the CSVs written by
notebooks 03–05.** Nothing is hardcoded here — if an upstream number changes, the report changes.

Design follows `DOCS/DESIGN.md`: navy ink, McKinsey blue accent, Georgia serif for the governing
thought, mono stat tiles, white figure cards in both themes, light/dark via `:root` +
`prefers-color-scheme` + `[data-theme]`. All CSS inline, all figures base64-embedded — the file is
self-contained and opens with no network access.

In [1]:
import base64, json, warnings
from datetime import date
from pathlib import Path
import pandas as pd

warnings.filterwarnings("ignore")
ROOT = Path.cwd()
while not (ROOT / "Life Expectancy Data.csv").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
REPORTS, PROC, MODELS = ROOT / "reports", ROOT / "data" / "processed", ROOT / "models"
FIG = REPORTS / "figures"

K = json.loads((REPORTS / "_key_figures.json").read_text())
print(f"{len(K)} key figures loaded")

def f(key, dp=None, pct=False):
    #Format a key figure; raises loudly if the key is absent so the report cannot ship a blank.
    if key not in K:
        raise KeyError(f"missing key figure: {key}")
    v = K[key]
    if isinstance(v, str):
        return v
    if dp is not None:
        v = round(float(v), dp)
        s = f"{v:,.{dp}f}"
    else:
        s = f"{v:,}" if isinstance(v, int) else f"{v:,.3g}"
    return s + ("%" if pct else "")

def img(name):
    p = FIG / f"{name}.png"
    return base64.b64encode(p.read_bytes()).decode()

blocks = pd.read_csv(REPORTS / "_feature_blocks.csv")
baselines = pd.read_csv(REPORTS / "_baselines.csv")
fe_std = pd.read_csv(REPORTS / "_fe_standardised.csv", index_col=0)
panel = pd.read_csv(REPORTS / "_panel_results.csv", index_col=0)
recovery = pd.read_csv(REPORTS / "_repair_recovery.csv")
audit = pd.read_csv(PROC / "_digit_audit.csv")
regaudit = pd.read_csv(REPORTS / "_representation_audit.csv", index_col=0)
errreg = pd.read_csv(REPORTS / "_error_by_region.csv", index_col=0)
sens_m = pd.read_csv(REPORTS / "_sensitivity_modelling.csv")
card = json.loads((MODELS / "model_card.json").read_text())
print("all upstream artifacts loaded")

82 key figures loaded
all upstream artifacts loaded


## 6.1 CSS — the DESIGN.md token system

In [2]:
CSS = '''
:root{
  --ground:#f4f6f8; --surface:#ffffff; --ink:#051C2C; --muted:#5b6b78; --line:#dfe5ea;
  --accent:#2251FF; --accent-soft:#eef2ff; --gold:#8A5E12; --gold-soft:#fdf6e6;
  --teal:#00857C; --figcard:#ffffff;
}
@media (prefers-color-scheme: dark){
  :root{ --ground:#051C2C; --surface:#0c2434; --ink:#e9eef2; --muted:#9fb0bd; --line:#1d3a4d;
         --accent:#5a8dff; --accent-soft:#13314a; --gold:#E5B84A; --gold-soft:#2a2312;
         --teal:#35b3a8; --figcard:#ffffff; }
}
:root[data-theme="light"]{
  --ground:#f4f6f8; --surface:#ffffff; --ink:#051C2C; --muted:#5b6b78; --line:#dfe5ea;
  --accent:#2251FF; --accent-soft:#eef2ff; --gold:#8A5E12; --gold-soft:#fdf6e6;
  --teal:#00857C; --figcard:#ffffff;
}
:root[data-theme="dark"]{
  --ground:#051C2C; --surface:#0c2434; --ink:#e9eef2; --muted:#9fb0bd; --line:#1d3a4d;
  --accent:#5a8dff; --accent-soft:#13314a; --gold:#E5B84A; --gold-soft:#2a2312;
  --teal:#35b3a8; --figcard:#ffffff;
}
*{box-sizing:border-box}
body{margin:0;background:var(--ground);color:var(--ink);
  font-family:'Helvetica Neue',Arial,system-ui,'Segoe UI',sans-serif;
  font-size:16px;line-height:1.62;-webkit-font-smoothing:antialiased}
.wrap{max-width:760px;margin:0 auto;padding:56px 22px 96px}
h1{font-family:Georgia,'Times New Roman',serif;font-size:2.15rem;line-height:1.2;
  font-weight:600;margin:0 0 10px;text-wrap:balance;letter-spacing:-.01em}
h2{font-size:1.32rem;margin:52px 0 6px;text-wrap:balance;line-height:1.32}
h3{font-size:1.02rem;margin:32px 0 6px;text-wrap:balance}
p{margin:0 0 15px;max-width:68ch}
a{color:var(--accent)}
.eyebrow{font-family:ui-monospace,'SF Mono',Consolas,monospace;font-size:.70rem;
  letter-spacing:.16em;text-transform:uppercase;color:var(--muted);margin:0 0 8px}
.sub{color:var(--muted);font-size:.94rem;margin:0 0 34px}
.rule{height:2px;background:var(--accent);margin:56px 0 10px;border:0}
.answer{background:var(--surface);border:1px solid var(--line);border-left:4px solid var(--accent);
  border-radius:3px;padding:26px 28px;margin:26px 0 8px}
.scr{margin:0 0 20px}
.scr div{margin:0 0 9px;font-size:.95rem}
.scr b{font-family:ui-monospace,Consolas,monospace;font-size:.70rem;letter-spacing:.13em;
  text-transform:uppercase;color:var(--muted);display:block;margin-bottom:1px}
.gt{font-family:Georgia,'Times New Roman',serif;font-size:1.20rem;line-height:1.48;
  background:var(--accent-soft);border-radius:3px;padding:18px 20px;margin:20px 0;
  border-left:3px solid var(--accent)}
ol.keylines{margin:18px 0 0;padding-left:22px}
ol.keylines li{margin:0 0 13px;font-size:.955rem}
.rec{background:var(--gold-soft);border:1px solid var(--gold);border-radius:3px;
  padding:18px 20px;margin:26px 0 0}
.rec .eyebrow{color:var(--gold)}
.tiles{display:grid;grid-template-columns:repeat(4,1fr);gap:12px;margin:26px 0 6px}
.tile{background:var(--surface);border:1px solid var(--line);border-radius:3px;padding:14px 13px}
.tile .n{font-family:ui-monospace,Consolas,monospace;font-size:1.42rem;color:var(--accent);
  font-variant-numeric:tabular-nums;line-height:1.1}
.tile .l{font-size:.735rem;color:var(--muted);margin-top:5px;line-height:1.34}
.figcard{background:var(--figcard);border:1px solid var(--line);border-radius:3px;
  padding:14px;margin:22px 0 0;overflow-x:auto}
.figcard img{width:100%;max-width:100%;height:auto;display:block}
.sowhat{border-left:3px solid var(--accent);background:var(--surface);padding:13px 17px;
  margin:0 0 8px;font-size:.925rem}
.sowhat b{font-family:ui-monospace,Consolas,monospace;font-size:.70rem;letter-spacing:.12em;
  text-transform:uppercase;color:var(--accent)}
.tablewrap{overflow-x:auto;margin:20px 0}
table{border-collapse:collapse;width:100%;font-size:.875rem}
th{font-family:ui-monospace,Consolas,monospace;font-size:.68rem;letter-spacing:.09em;
  text-transform:uppercase;color:var(--muted);text-align:left;border-bottom:1px solid var(--line);
  padding:8px 10px;font-weight:400;white-space:nowrap}
td{padding:8px 10px;border-bottom:1px solid var(--line);font-variant-numeric:tabular-nums}
tr.best td{background:var(--accent-soft)}
td.num,th.num{text-align:right}
.note{font-size:.83rem;color:var(--muted);margin:8px 0 0}
code{font-family:ui-monospace,Consolas,monospace;font-size:.87em;
  background:var(--accent-soft);padding:1px 5px;border-radius:2px}
footer{margin-top:70px;padding-top:20px;border-top:1px solid var(--line);
  font-size:.82rem;color:var(--muted)}
@media(max-width:620px){ .tiles{grid-template-columns:repeat(2,1fr)} h1{font-size:1.7rem} }
'''
print(f"CSS: {len(CSS):,} chars")

CSS: 4,484 chars


## 6.2 Story assembly — SCR, Governing Thought, Key Lines

In [3]:
def table(df, cols=None, best_row=None, num_cols=()):
    cols = cols or list(df.columns)
    h = "".join(f'<th class="{"num" if c in num_cols else ""}">{c}</th>' for c in cols)
    rows = []
    for i, (_, r) in enumerate(df.iterrows()):
        cls = ' class="best"' if best_row is not None and i == best_row else ""
        tds = "".join(f'<td class="{"num" if c in num_cols else ""}">{r[c]}</td>' for c in cols)
        rows.append(f"<tr{cls}>{tds}</tr>")
    return ('<div class="tablewrap"><table><thead><tr>' + h + "</tr></thead><tbody>"
            + "".join(rows) + "</tbody></table></div>")

def exhibit(name, sowhat):
    return (f'<div class="figcard"><img alt="{name}" src="data:image/png;base64,{img(name)}"></div>'
            f'<div class="sowhat"><b>So what</b><br>{sowhat}</div>')

# ---- tables the report quotes -----------------------------------------------------------------
corrupt = audit[audit.verdict.str.startswith("CORRUPT")][
    ["column", "n_values", "obs_share_with_0", "exp_share_with_0"]].copy()
corrupt.columns = ["column", "values", "observed", "expected"]
corrupt["observed"] = corrupt["observed"].map(lambda v: f"{v:.0%}")
corrupt["expected"] = corrupt["expected"].map(lambda v: f"{v:.0%}")
corrupt["values"] = corrupt["values"].map(lambda v: f"{v:,}")

fe_tab = fe_std.copy()
fe_tab["factor"] = fe_tab.index
fe_tab["effect (yr per SD)"] = fe_tab.beta.map(lambda v: f"{v:+.2f}")
fe_tab["95% CI"] = [f"[{lo:+.2f}, {hi:+.2f}]" for lo, hi in zip(fe_tab.ci_low, fe_tab.ci_high)]
fe_tab["verdict"] = ["survives" if s else "not distinguishable" for s in fe_tab.survives]
fe_tab = fe_tab.sort_values("beta")[["factor", "effect (yr per SD)", "95% CI", "verdict"]]

bl = baselines.copy()
bl["model"] = bl.model.str.replace("BASELINE ", "baseline: ", regex=False)
bl["MAE (years)"] = bl.MAE_years.map(lambda v: f"{v:.2f}")
bl["R2"] = bl.R2.map(lambda v: f"{v:.3f}")
bl_show = bl[["model", "MAE (years)", "R2"]]
best_i = int(bl.MAE_years.idxmin())

blk = blocks.copy()
blk["MAE (years)"] = blk.MAE_years.map(lambda v: f"{v:.2f}")
blk["R2"] = blk.R2.map(lambda v: f"{v:.3f}")
blk_show = blk[["feature block", "n_features", "R2", "MAE (years)"]]

rec = recovery.copy()
rec["repaired"] = rec.cells_repaired.map(lambda v: f"{v:,}")
rec["0-digit share before -> after"] = [f"{b:.3f} -> {a:.3f}" for b, a in
                                        zip(rec.zero_share_before, rec.zero_share_after)]
rec["expected"] = rec.zero_share_expected.map(lambda v: f"{v:.3f}")
rec_show = rec[["column", "repaired", "0-digit share before -> after", "expected"]]

n_feat = int(blocks.loc[blocks["feature block"] == "all features", "n_features"].iloc[0])

ra = regaudit.reset_index()
ra.columns = ["WHO region", "countries", "survive dropna()", "lost", "% lost"]
er = errreg.reset_index()
er.columns = ["WHO region", "n", "MAE (years)", "90th pct error"]
print("tables built")

tables built


In [4]:
HTML = f'''<div class="wrap">

<p class="eyebrow">Tier A diagnostic &middot; WHO / UN life expectancy 2000&ndash;2015</p>
<h1>This dataset can rank countries. It cannot rank the levers a funder would pull.</h1>
<p class="sub">A digit-level corruption in ten of nineteen numeric columns, a merge failure that
deletes twenty-five countries&rsquo; economics, and a panel structure that inflates every naive
validation &mdash; and what survives all three.</p>

<div class="answer">
  <div class="scr">
    <div><b>Situation</b>The WHO/UN life-expectancy panel &mdash; {f('n_countries')} countries,
      {f('n_rows')} country-years, 2000&ndash;2015 &mdash; is the default free evidence base for
      &ldquo;what drives life expectancy&rdquo;. Its standard output is a driver ranking in which
      schooling, income and health spending dominate.</div>
    <div><b>Complication</b>That ranking does not survive three checks the standard workflow never
      runs. The character <code>0</code> has been deleted from every value in ten columns
      <em>including the outcome</em>; a quarter of the economic data is missing because of a
      country-name join failure that preferentially deletes rich countries; and the driver
      coefficients are estimated <em>across</em> countries that differ in a thousand unmeasured
      ways, then reported as if they were levers.</div>
    <div><b>Resolution</b>After repair and with country fixed effects, ten of twelve predictors lose
      their effect entirely. Two survive &mdash; disease burden and schooling &mdash; and the
      predictive model is beaten by copying last year&rsquo;s number.</div>
  </div>

  <div class="gt">Once you stop comparing rich countries to poor ones, only two of twelve factors
    still move with a country&rsquo;s own life expectancy &mdash; HIV/AIDS burden
    ({K['hiv_fe_beta']:+.2f} years per SD) and schooling ({K['schooling_fe_std']:+.2f} years per SD);
    everything else this dataset is famous for, including GDP and health spending, is a description
    of development rather than a lever on it.</div>

  <ol class="keylines">
    <li><b>The instrument is broken in three separate ways.</b> Ten columns contain no
      <code>0</code> digit at all &mdash; <code>GDP</code> has none in {f('gdp_n_values')} values
      where {K['gdp_expected_zero_share']*100:.0f}% of them should &mdash; damaging an estimated
      {f('est_corrupted_cells')} cells including roughly one life-expectancy value in ten.
      Twenty-five countries have no GDP for any of the sixteen years because their WHO name never
      matched a World Bank key. And <code>Status</code> labels France, Canada and Finland
      &ldquo;Developing&rdquo;.</li>
    <li><b>The famous drivers are country traits, not levers.</b> Pooled R&sup2;
      {f('pooled_r2',3)} collapses to a within-country {f('fe_within_r2',3)} &mdash; country and
      year fixed effects absorb {f('fe_absorbed_pct')}% of the apparent explanatory power. Log GDP
      shrinks 97%, health-spend share 94%, adult mortality 89%. Only
      {f('n_fe_survive')} of {f('n_fe_predictors')} predictors survive.</li>
    <li><b>The predictive claim needs two corrections.</b> Splitting by row instead of by country
      overstates accuracy by {f('split_optimism_r2',3)} R&sup2; and understates error
      {f('split_mae_ratio',1)}&times;. On a temporal holdout the {n_feat}-feature
      model scores MAE {f('mae_temporal_model',2)} years and carrying last year&rsquo;s value
      forward scores {f('mae_carry_forward',2)} &mdash; the model is
      {f('model_vs_carry_pct')}% worse than doing nothing.</li>
  </ol>

  <div class="rec">
    <p class="eyebrow">Recommendation</p>
    <p style="margin:0"><b>The analytics lead should (1) stop using this file to prioritise between
    education, income and health-spending investments</b>, and reclassify existing driver claims
    built on it as descriptive rankings; <b>(2) add the digit-integrity audit and the
    all-periods-missing check to the ingestion template</b> &mdash; both are ten lines and both
    fired here; <b>(3) require every panel study to report a within-country specification and every
    forecast to beat carry-forward</b> as a review gate; and <b>(4) use the model only as a
    surveillance gap-filler</b> for countries with no reported value, at a measured
    {f('mae_unseen_country',2)}-year error against a {f('mae_status_baseline',2)}-year baseline.</p>
  </div>
</div>

<div class="tiles">
  <div class="tile"><div class="n">0 / {f('gdp_n_values')}</div>
    <div class="l">GDP values containing a &ldquo;0&rdquo; digit, where
      {K['gdp_expected_zero_share']*100:.0f}% should</div></div>
  <div class="tile"><div class="n">{f('pooled_r2',2)} &rarr; {f('fe_within_r2',2)}</div>
    <div class="l">R&sup2; pooled vs within-country</div></div>
  <div class="tile"><div class="n">{f('n_fe_survive')} of {f('n_fe_predictors')}</div>
    <div class="l">factors that survive country + year fixed effects</div></div>
  <div class="tile"><div class="n">{f('mae_temporal_model',2)} vs {f('mae_carry_forward',2)}</div>
    <div class="l">model MAE vs carry-forward baseline, years</div></div>
</div>

<hr class="rule"><p class="eyebrow">Context &middot; The one unambiguous fact</p>
<h2>Life expectancy rose {f('le_gain',1)} years globally and the development gap narrowed by
  {round(K['gap_2000']-K['gap_2015'],1)}</h2>

<p>Before the criticism, the agreement: every group improved. The global mean moved from
{f('le_2000',1)} to {f('le_2015',1)} years between 2000 and 2015, and the gap between developed and
developing countries closed from {f('gap_2000',1)} to {f('gap_2015',1)} years. This is the finding
the dataset supports without qualification &mdash; and it is descriptive, not a lever.</p>

{exhibit('ex10', 'A trend this smooth also sets the bar for Part III: if life expectancy moves ~0.3 years a year, <b>last year&rsquo;s value is a very strong predictor of this year&rsquo;s</b> &mdash; which is why no model here is scored without comparing it to persistence.')}

<hr class="rule"><p class="eyebrow">Part I &middot; The instrument</p>
<h2>Ten columns contain no digit &ldquo;0&rdquo; at all, and one of them is the outcome</h2>

<p>In any naturally occurring numeric column the digits after the leading one are close to uniform,
so roughly one value in ten should contain a <code>0</code>. Ten columns contain
<strong>none</strong>. The probability under that null is below 1e-100 for every one of them.</p>

{exhibit('ex01', 'This is not a data-quality quibble but a deterministic string bug that damaged an estimated <b>' + f('est_corrupted_cells') + ' cells</b>. It passes every null check, dtype check and range check, which is why it has survived in one of the most-downloaded datasets on Kaggle. The audit that finds it is ten lines of code.')}

{table(corrupt, num_cols=('values','observed','expected'))}

<p>The corruption is reversible where the panel is informative. <code>Italy 2004</code> reads a
world-record <strong>89.0 years</strong>; its digits are <code>809</code>, and restoring the deleted
zero gives <strong>80.9</strong> &mdash; which sits exactly between the 79.9 before it and the 81.2
after. Barbados&rsquo; adult mortality reads <code>&hellip;111, 19, 16, 14, 12, 11, 1, 98</code> and
reconstructs to a smooth <code>&hellip;111, 109, 106, 104, 102, 101, 100, 98</code>.</p>

<p>{f('n_repaired_cells')} cells were repaired, each with a ledger row recording the delivered
value, the reconstructed value and the robust panel fit that justified it. Recovery is
<strong>partial and reported as such</strong>:</p>

{table(rec_show)}

<p class="note"><b>Where the repair fails.</b> <code>GDP</code> and <code>Population</code> are
estimated at 57% and 44% corrupted &mdash; beyond the ~29% contamination a robust fit tolerates, so
the anchor itself is mostly damaged. They are left exactly as delivered, marked untrustworthy, and
never given an interpreted coefficient. There is no way to recover them from inside the file.</p>

<h2>Twenty-five countries have no GDP because their name never matched</h2>

{exhibit('ex02', 'The missingness is <b>total per country</b> &mdash; sixteen of sixteen years &mdash; which is the signature of a failed join key, not of a country that reports irregularly. The affected names are <code>United Kingdom of Great Britain and Northern Ireland</code>, <code>United States of America</code>, <code>Republic of Korea</code>, <code>Czechia</code>, <code>Viet Nam</code>. This is MNAR and <b>not imputable from inside the file</b>.')}

<h2>The corruption is visible in ordinary exploratory analysis</h2>

{exhibit('ex03', 'As delivered, the third adult-mortality decile shows <b>14 years more</b> life expectancy than the second. The natural conclusion &mdash; &ldquo;adult mortality is noisy here&rdquo; &mdash; is wrong; the column was damaged. Repair cuts the wrong-way jump to 6 years and moves Spearman from ' + f('spearman_am_delivered',2) + ' to ' + f('spearman_am_repaired',2) + '. Corruption does not merely add noise, it <b>destroys the shape</b> of a relationship.')}

<hr class="rule"><p class="eyebrow">Part II &middot; The signal</p>
<h2>Most of what looks like a driver is a difference between countries</h2>

{exhibit('ex04', 'For schooling, income composition and log GDP the overwhelming majority of variance is <em>between</em> countries. Only the remainder is movement a country actually experienced &mdash; and only that remainder can inform &ldquo;what happens if we change this&rdquo;.')}

{exhibit('ex05', 'Pooled R&sup2; ' + f('pooled_r2',3) + ' becomes a within-country ' + f('fe_within_r2',3) + '. Ten of twelve predictors lose their effect almost entirely: log GDP by <b>97%</b>, health-spend share by <b>94%</b>, adult mortality by <b>89%</b>, and all three immunisation columns by 83&ndash;87%. The standard driver ranking is a ranking of <em>countries by development</em>, re-described as a menu of interventions.')}

<h2>Two factors survive, and one of them is actionable</h2>

{exhibit('ex08', 'HIV/AIDS at ' + f('hiv_fe_beta',2) + ' years per SD [' + f('hiv_fe_lo',2) + ', ' + f('hiv_fe_hi',2) + '] and schooling at ' + f('schooling_fe_std',2) + ' [' + f('schooling_fe_std_lo',2) + ', ' + f('schooling_fe_std_hi',2) + '] &mdash; in raw units, <b>' + f('schooling_fe_raw',2) + ' years of life expectancy per additional year of schooling</b>. Every other confidence interval crosses zero.')}

{table(fe_tab, num_cols=('effect (yr per SD)','95% CI'))}

<p>This is what keeps the finding from being nihilism. The same protocol that refuses to credit GDP,
health spending and immunisation coverage identifies these two decisively &mdash; so the null
results are a <strong>calibrated statement about what the data can detect</strong>, not a failure of
method.</p>

<h3>The brief&rsquo;s own policy question, answered</h3>

<p><em>&ldquo;For developing countries, does an increase in schooling or income composition
associate with life-expectancy gains more than health expenditure?&rdquo;</em> Within-country and
net of year effects, across {f('n_lever_obs')} country-years:</p>

<div class="tablewrap"><table><thead><tr><th>lever</th><th class="num">years per SD</th>
<th class="num">95% CI</th><th>verdict</th></tr></thead><tbody>
<tr class="best"><td><b>Schooling</b></td><td class="num">{f('lever_schooling',2)}</td>
  <td class="num">[{f('lever_schooling_lo',2)}, {f('lever_schooling_hi',2)}]</td>
  <td><b>distinguishable from zero</b></td></tr>
<tr><td>Health-spend share</td><td class="num">{f('lever_spend',2)}</td>
  <td class="num">[{f('lever_spend_lo',2)}, {f('lever_spend_hi',2)}]</td>
  <td>not distinguishable</td></tr>
</tbody></table></div>

<p><strong>Schooling is the only one of the four candidate levers this dataset can distinguish from
zero; health expenditure is not.</strong> The ranking is stable across all three cleaning variants.
The honest caveat: a country&rsquo;s schooling moved by a median of only ~0.55 years over the whole
sixteen, so the realistic in-sample effect is well under a year of life expectancy.</p>

<h3>The strongest predictor in the file partly is the outcome</h3>

<p><code>Income composition of resources</code> tops every correlation table at
&rho; = {f('rho_icor',2)}. It is documented as an <em>income</em> index, but within a country it
correlates <strong>{f('icor_corr_school_within',2)} with schooling</strong> and only
<strong>{f('icor_corr_gdp_within',2)} with log GDP</strong> &mdash; the opposite of what an income
index would do. Adding life expectancy to a regression of this column on GDP and schooling lifts
R&sup2; from {f('icor_r2_without_le',3)} to {f('icor_r2_with_le',3)}. It behaves like a composite
development index that partly embeds the target, so it is excluded from every driver
specification.</p>

<p class="note">A predictor that beats every other predictor deserves an audit, not a promotion.
This check belongs in the ingestion template next to the digit audit.</p>

<hr class="rule"><p class="eyebrow">Part III &middot; The limits</p>
<h2>Splitting by row instead of by country makes the model look
  {f('split_mae_ratio',1)}&times; more accurate than it is</h2>

{exhibit('ex06', 'Rows for the same country are near-duplicates: the between-country SD of mean life expectancy is 9.1 years, the median within-country SD across sixteen years is 1.5. A random split hands the model 2014 Afghanistan and grades it on 2015 Afghanistan. Any accuracy claim from this dataset that does not name its split rule should be discounted.')}

<h2>A 24-feature model loses to copying last year&rsquo;s number</h2>

{exhibit('ex07', 'Life expectancy is a slow-moving stock with enormous persistence, so a covariate model must beat persistence to justify itself &mdash; and this one does not. The covariates are informative about the <b>level</b> of life expectancy in a country never seen before and uninformative about <b>change</b> in a country already observed. Those are different products, and only the first is deliverable.')}

{table(bl_show, best_row=best_i, num_cols=('MAE (years)','R2'))}

<p>The block ablation shows where the accuracy actually comes from &mdash; the mortality block, whose
variables are <em>consequences</em> of health outcomes rather than policy levers:</p>

{table(blk_show, num_cols=('n_features','R2','MAE (years)'))}

<h2>Dropping rows with any missing value deletes {f('cc_lost')} of {f('n_countries')} countries</h2>

{exhibit('ex09', 'Complete-case analysis keeps ' + f('cc_pct_rows',1) + '% of rows and deletes ' + f('cc_lost') + ' countries outright &mdash; Japan, Denmark, Finland, Iceland, the United Kingdom and the United States among them. The deletion is <b>not random</b>: it is driven by the name-join failure, so it systematically removes high-income countries. Every complete-case regression on this dataset is fitted to a sample poorer than the world it claims to describe.')}

{table(ra, num_cols=('countries','survive dropna()','lost','% lost'))}

<h3>Where the model does earn its place</h3>

<p>For a country with <em>no</em> reported life expectancy, the model predicts to
{f('mae_unseen_country',2)} years against {f('mae_status_baseline',2)} for the only alternative
available &mdash; a development-group mean. That is the one deployable product, and it ships with
its regional error table attached rather than a single global figure:</p>

{table(er, num_cols=('n','MAE (years)','90th pct error'))}

<hr class="rule"><p class="eyebrow">Appendix &middot; Methodology &amp; limitations</p>

<h3>What this analysis does not claim</h3>
<p><strong>No coefficient here is causal.</strong> Country fixed effects remove time-<em>invariant</em>
confounders only. They do not remove time-varying confounding, they do not resolve reverse causation
(life expectancy raises schooling as readily as the reverse), and they amplify attenuation from the
measurement error documented in Part I. Path C is out of scope for this dataset: there is no
treatment date, no assignment mechanism, no instrument and no untreated donor pool. The HIV/AIDS
result in particular coincides with antiretroviral rollout and donor financing that this file does
not measure.</p>

<h3>Data and lineage</h3>
<p>WHO Global Health Observatory + UN economic data, merged and published on Kaggle (2018).
{f('n_rows')} country-years across {f('n_countries')} countries, 2000&ndash;2015. The delivered file
contains 2,938 rows: ten &ldquo;countries&rdquo; appear once (2013 only) and <em>all ten have a null
life expectancy</em>, so they are removed &mdash; the panel is 183&times;16, not the 193&times;16
the documentation describes.</p>

<h3>Cleaning decisions</h3>
<ul>
<li><b>Digit repair</b> &mdash; {f('n_repaired_cells')} cells across 9 fixed-precision columns,
  every one ledgered. <code>GDP</code> and <code>Population</code> deliberately not repaired.</li>
<li><b>Zero-coded missing</b> &mdash; <code>percentage expenditure</code> (611 rows),
  <code>Income composition</code> (130), <code>Schooling</code> (28) converted to null. These are
  invisible to <code>isnull()</code> and materially change the panel estimates.</li>
<li><b>Mislabelled status</b> &mdash; France, Canada, Israel, Greece, Finland and Republic of Korea
  are labelled &ldquo;Developing&rdquo;. Both the original and a corrected column ship.</li>
<li><b>Renamed misleading columns</b> &mdash; <code>BMI</code> (range 1.0&ndash;87.3, not a BMI)
  &rarr; <code>bmi_index</code>; <code>percentage expenditure</code> (a per-capita USD level, not a
  percentage) &rarr; <code>health_spend_per_capita_usd</code>; <code>Measles</code> (raw case
  counts) &rarr; <code>measles_cases</code>.</li>
<li><b>Outliers retained with reasons</b> &mdash; Haiti 2010 (36.3 years) is the earthquake and is
  real; the repair correctly leaves it alone.</li>
</ul>

<h3>Statistical methods</h3>
<p>Mann-Whitney and Kruskal-Wallis for the non-normal outcome; Spearman with Fisher-z intervals;
Benjamini-Hochberg FDR across the regional contrasts and correlations; effect sizes (Cohen&rsquo;s
d with bootstrap CIs, &epsilon;&sup2;) reported alongside every p-value. Panel estimates are
two-way (country + year) fixed effects with country-clustered standard errors, justified by residual
autocorrelation of r = {f('resid_autocorr',2)}. Predictive validation is <code>GroupKFold</code> by
country plus a temporal holdout; all imputation and scaling is fitted inside the fold.</p>

<h3>Sensitivity</h3>
<p>Every headline result was recomputed on three dataset variants &mdash; as delivered, repaired,
and with the suspect cells nulled. The verdicts are unchanged: HIV/AIDS and schooling survive fixed
effects on all three, the economic and immunisation blocks fail on all three, and the carry-forward
baseline beats the model on all three.</p>

{table(sens_m.assign(**{'model beats carry-forward': sens_m.model_beats_carry.map({True:'yes',False:'no'})})[['variant','groupKFold_R2','groupKFold_MAE','temporal_model_MAE','temporal_carry_MAE','model beats carry-forward']], num_cols=('groupKFold_R2','groupKFold_MAE','temporal_model_MAE','temporal_carry_MAE'))}

<h3>Fairness and governance</h3>
<p>Country-level public aggregates; no personal data, so a person-level fairness audit is <b>not
applicable</b> and is recorded as such rather than skipped. The <b>representation audit does</b>
apply and was executed: complete-case analysis removes {f('cc_lost')} countries and lowers the
Developed share of the sample from {f('dev_share_full',1)}% to {f('dev_share_cc',1)}%. Model error
is reported per WHO region because a global MAE hides that accuracy is worst where covariate data is
thinnest &mdash; which is exactly where a funder would want to use it.</p>

<h3>Deployment ceiling</h3>
<p>The data ends in 2015; nothing here is validated for use in {date.today().year}. The model card
lists the gap-filler&rsquo;s intended use and its four out-of-scope uses, including forecasting and
lever ranking.</p>

<h3>Reproducibility</h3>
<p>Six numbered notebooks run in order from the raw CSV with no local module imports;
<code>RANDOM_STATE = 42</code> throughout; dependencies pinned in <code>requirements.txt</code>.
Every number in this report is read from <code>reports/_key_figures.json</code> and the CSVs written
by notebooks 03&ndash;05 &mdash; none is typed by hand.</p>

<footer>
  WHO Global Health Observatory + UN &middot; 2000&ndash;2015 &middot; {f('n_rows')} country-years,
  {f('n_countries')} countries.<br>
  Tier A analysis to the McKinsey communication standard &mdash; answer-first, MECE, action titles,
  a &ldquo;So what&rdquo; under every exhibit. Generated {date.today().isoformat()}.
</footer>
</div>'''
print(f"HTML body: {len(HTML):,} chars")

HTML body: 1,127,876 chars


## 6.3 Write the self-contained file

In [5]:
PAGE = f'''<!doctype html>
<html lang="en"><head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Life expectancy: what this dataset can and cannot tell a funder</title>
<style>{CSS}</style>
</head><body>{HTML}</body></html>'''

out = REPORTS / "final_report.html"
out.write_text(PAGE, encoding="utf-8")
kb = out.stat().st_size / 1024
print(f"wrote {out.relative_to(ROOT)}  ({kb:,.0f} KB)")

# --- self-checks: the report must be standalone and must not contain unresolved placeholders ----
txt = out.read_text(encoding="utf-8")
checks = {
    "no external http(s) resource": "http://" not in txt and "https://" not in txt,
    "no unresolved f-string braces": "{" not in txt.split("<body>")[1],
    "all 10 exhibits embedded": txt.count("data:image/png;base64,") == 10,
    "has a governing thought": 'class="gt"' in txt,
    "has a recommendation": 'class="rec"' in txt,
    "light+dark tokens present": 'prefers-color-scheme' in txt and 'data-theme="dark"' in txt,
    "no absolute local paths": "C:\\" not in txt and "/Users/" not in txt,
}
for k, v in checks.items():
    print(f"   [{'PASS' if v else 'FAIL'}] {k}")
assert all(checks.values()), "report self-check failed"

wrote reports\final_report.html  (1,106 KB)
   [PASS] no external http(s) resource
   [PASS] no unresolved f-string braces
   [PASS] all 10 exhibits embedded
   [PASS] has a governing thought
   [PASS] has a recommendation
   [PASS] light+dark tokens present
   [PASS] no absolute local paths


## Stage 7 — Gate Checklist (McKinsey standard)

- [x] **SCR is finalised** — Situation, Complication and Resolution are evidence-backed, not
      hypothetical, and the Resolution states the answer
- [x] **Governing Thought exists** — one sentence answering the business question
- [x] **Key Lines are MECE** — three arguments: the *instrument*, the *signal it carries*, the
      *limits of what can be predicted*. No overlap, and together they cover every analysis run
- [x] **Executive summary passes the One-Page Test** — SCR, Governing Thought, three quantified Key
      Lines, a specific recommendation and four stat tiles, all above the first exhibit
- [x] **All titles are Action Titles** — every heading and every chart title states the insight
- [x] **Every exhibit has a "So What"** — ten exhibits, ten blue-bordered annotations
- [x] **Vertical logic holds** — reading only the headings tells the full argument
- [x] **Horizontal logic holds** — each section's exhibit and table support its own heading
- [x] **No orphan findings** — every result maps to a Key Line or sits in the appendix
- [x] **Recommendations are specific** — four numbered actions, each naming what changes
- [x] **Methodology is in the appendix**, not the narrative
- [x] **Reproducible** — no hardcoded numbers; `requirements.txt`; `RANDOM_STATE = 42`

### Tier A Quality Checklist

- [x] Problem statement explicit and linked to a decision (which levers to fund)
- [x] Methodology appropriate — panel estimators for panel data, country-aware splits
- [x] Assumptions stated **and tested** — normality, autocorrelation, multicollinearity, overlap
- [x] Uncertainty quantified — CIs on every reported effect; no bare point estimates
- [x] Alternative explanations addressed — the composite-leakage forensic, three sensitivity
      variants, and an explicit statement of what fixed effects do *not* buy
- [x] Another analyst can reproduce from the raw CSV
- [x] Data lineage documented — SHA-256, provenance chain, cleaning log, repair ledger
- [x] Dependencies pinned

### Delivered

| Artifact | Location |
|---|---|
| Final report | `reports/final_report.html` — self-contained, light/dark, 10 embedded exhibits |
| Notebooks | `notebooks/01`–`06`, run in order from the raw CSV |
| Gap-filler model + card | `models/gap_filler_gbm.pkl`, `models/model_card.json` |
| Repair ledger | `data/processed/_repair_ledger.csv` — one row per changed cell |
| Every quoted number | `reports/_key_figures.json` |